# 내 목소리로 Supertonic 스타일 만들기 (Voice Builder 없이)

보이스 리더 앱에서 쓸 목소리 스타일 JSON을 **내 녹음**으로 만듭니다. 방법과 한계는 저장소의 `spike/voice_inversion/README.md`에 있습니다.

- **런타임 → 런타임 유형 변경 → GPU(T4)** 를 고르고 위에서부터 차례로 실행하세요.
- 녹음은 **내 구글 드라이브와 이 코랩 세션에만** 있습니다. 이 노트북은 녹음을 다른 서버로 보내지 않습니다. 세션을 닫으면 코랩 쪽 사본은 사라집니다.
- **본인 목소리이거나, 녹음된 사람에게 동의를 받은 목소리만** 쓰세요. Supertonic 가중치 라이선스(OpenRAIL-M, Attachment A)는 동의 없이 다른 사람을 흉내 내는 데 쓰는 것을 금지합니다.

## 1. 동의 확인

아래 문장을 읽고, 맞으면 `CONSENT = True`로 바꾼 뒤 실행하세요. 동의 문장을 녹음 파일 하나로 남겨 두는 것도 권합니다(`consent.wav`).

> 나는 이 녹음 속 목소리의 주인이며(또는 주인에게 동의를 받았으며), 이 목소리로 음성 합성 스타일을 만들어 내 기기에서 책을 읽는 데 쓰는 것에 동의합니다.

In [ ]:
CONSENT = False  # 위 문장에 동의하면 True
assert CONSENT, "동의 문장을 확인하고 CONSENT = True로 바꾸세요"

## 2. 준비

드라이브에 폴더 하나(기본 `MyDrive/voice-reader/recordings`)를 만들고 녹음 wav를 넣어 두세요. 결과는 `MyDrive/voice-reader/out`에 저장됩니다.

녹음 요령: 조용한 방, 입에서 한 뼘 거리, 평소 책 읽는 속도와 톤으로 한국어 문장을 읽습니다. **합쳐서 1분 안팎 권장, 최소 15초**(검증은 12초까지만 했습니다). 한 파일은 5~30초. 잡음 제거·음악·효과는 넣지 마세요.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
REC_DIR = '/content/drive/MyDrive/voice-reader/recordings'
OUT_DIR = '/content/drive/MyDrive/voice-reader/out'
NAME = 'me'                 # 결과 파일 이름 (영문/숫자)
MODEL = 'supertonic-2'      # 앱의 서술 목소리. 감정 문장용이면 'supertonic-3' (모델마다 따로 만들어야 합니다)
import os, glob
os.makedirs(OUT_DIR, exist_ok=True)
wavs = sorted(glob.glob(f'{REC_DIR}/*.wav'))
print(len(wavs), 'files'); assert wavs, 'REC_DIR에 wav가 없습니다'

이 노트북은 GitHub `MJ-best/voice-reader`의 `spike/voice_inversion/`을 받아 씁니다. 그 폴더가 main에 올라가 있어야 합니다.

In [ ]:
%%capture
!pip install onnx onnx2torch onnxruntime speechbrain sherpa-onnx==1.13.8 soundfile
!git clone --depth 1 https://github.com/MJ-best/voice-reader /content/voice-reader

In [ ]:
import os
os.environ['VI_CACHE'] = '/content/vi_cache'   # 모델·가중치 캐시(세션과 함께 사라짐)
import soundfile as sf, numpy as np, torch, torchaudio.functional as AF
# 녹음을 44.1kHz 모노 하나로 이어 붙인다 (파일 사이 0.3초 무음)
parts = []
for f in wavs:
    w, sr = sf.read(f, dtype='float32', always_2d=True)
    parts += [AF.resample(torch.from_numpy(w.mean(1)), sr, 44100).numpy(), np.zeros(13230, np.float32)]
ref = np.concatenate(parts)
sf.write('/content/ref.wav', ref, 44100)
print(f'{len(ref) / 44100:.1f} s')

## 3. 스타일 역추정

맥 M4에서 약 12분(반복 200회) 걸렸습니다. 코랩 T4 시간은 아직 재 보지 않았습니다.

- 프리셋 혼합에서 시작해 `ecapa` 유사도를 올리고, 25회마다 검증 문장으로 잰 `wavlm` 유사도가 가장 높았던 시점을 `invert`로 저장합니다.
- `wavlm`이 시작값보다 한 번도 안 오르면 `invert`는 혼합과 같습니다. 그때는 녹음을 늘리거나 다시 녹음하세요.

In [ ]:
!cd /content/voice-reader && python spike/voice_inversion/voice_inversion.py fit \
    --ref /content/ref.wav --name {NAME} --lang ko --model {MODEL} --out {OUT_DIR}

## 4. 음성 팩 만들고 들어 보기

`pack`은 프리셋 10개(sid 0~9) 뒤에 새 스타일을 붙여 앱 음성 팩 zip을 만듭니다. 앱에는 스타일 JSON만 가져오는 쪽(5번)이 간단합니다.

`eval`은 sherpa-onnx(앱과 같은 int8 모델)로 합성해 내 녹음과의 유사도(`wespeaker`, `wavlm`; 클수록 비슷)와 받아쓰기 오류율(`ko_cer`)을 보여 줍니다. `invert`의 `ko_cer`이 0.1을 넘거나 `wespeaker`가 가장 좋은 프리셋보다 낮으면 `blend`나 프리셋을 쓰세요. 숫자보다 **귀로 들어 보는 것**이 먼저입니다.

In [ ]:
!cd /content/voice-reader && python spike/voice_inversion/voice_inversion.py pack \
    {OUT_DIR}/{NAME}.nearest.json {OUT_DIR}/{NAME}.blend.json {OUT_DIR}/{NAME}.invert.json \
    --model {MODEL} --out {OUT_DIR}/{NAME}-{MODEL}.zip
!cd /content/voice-reader && python spike/voice_inversion/voice_inversion.py eval {OUT_DIR}/{NAME}-{MODEL}.zip \
    --ref /content/ref.wav --name {NAME} --out /content/eval

In [ ]:
import json, glob
from IPython.display import Audio, display
r = json.load(open('/content/eval/eval.json'))[NAME]
for k, v in r.items():
    print(k, v if not isinstance(v, dict) else {kk: vv for kk, vv in v.items() if kk != 'same_text'})
for f in sorted(glob.glob('/content/eval/*invert*__1.wav')) + sorted(glob.glob('/content/eval/*nearest*__1.wav')):
    print(f); display(Audio(f))

## 5. 앱에 넣기 (사용 방법 8번)

1. 귀로 들어 보고 `invert`와 `blend` 중 나은 쪽을 고릅니다. 파일 이름이 앱 목록에 그대로 보이니 `내 목소리.json`처럼 바꿔 두세요.
2. 파일을 폰으로 옮기고, 앱의 **목소리** 버튼 → **직접 만든 목소리 넣기** → 동의 확인 → **만든 모델과 같은 음성 팩**(`MODEL`이 supertonic-2면 Supertonic 2)을 고릅니다.
3. 목록 맨 끝에 추가된 목소리를 눌러 들어 봅니다.
4. 다 쓰면 드라이브의 `recordings` 폴더를 지워도 됩니다. 스타일 JSON에는 녹음이 들어 있지 않지만 그 목소리를 흉내 낼 수 있으니 공유하지 마세요.